# Lorenz Water Wheel — Evaluate & compare (Colab)

A **test run**: no training. Clones the repo (which carries the committed weights
under `weights/`) and scores each model on `data/test-dataset` with the **same**
metrics, so the mean `|ω|` RMSE numbers are directly comparable.

Needs both TensorFlow (LSTM) and torch (TCN) — both are preinstalled on Colab. A GPU
runtime is optional but faster.

## 1. Get the code + committed weights

In [ ]:
# --depth 1: grab only the latest snapshot, not the full history (faster clone).
!git clone --depth 1 https://github.com/antmatrik/Lorenz-wheel-prediction-LSTM.git
%cd Lorenz-wheel-prediction-LSTM

## 2. Install dependencies (both frameworks)

In [ ]:
!pip install -q -r requirements/eval.txt

## 3. Which weights are committed?

In [ ]:
!ls -la weights/

## 4. Score each model from `weights/`

Whichever model has no committed weights is skipped. The TCN uses its native
3-channel rollout; pass `--physics` to score it through the LSTM-style rollout
instead (isolates the architecture from the rollout strategy). Metrics are computed
identically for both, and also written to `outputs/eval_results.csv`.

In [ ]:
import os

def run(cmd):
    print('\n$', cmd, flush=True)
    get_ipython().system(cmd)

LSTM = ('--model lstm --weights weights/lstm.weights.h5 --stats weights/lstm_stats.npz')
TCN  = ('--model tcn --checkpoint weights/tcn_checkpoint.pt')

if os.path.exists('weights/lstm.weights.h5'):
    run(f'python src/common/evaluate.py {LSTM}')
else:
    print('skip LSTM: weights/lstm.weights.h5 not committed yet')

if os.path.exists('weights/tcn_checkpoint.pt'):
    run(f'python src/common/evaluate.py {TCN}')
else:
    print('skip TCN: weights/tcn_checkpoint.pt not committed yet')

## 5. Skill vs horizon (both models)

One rollout to the largest horizon, reported at each cutoff, with naive baselines
(`base0`, `persist`). Compare `|w|RMSE` (magnitude) across models.

In [ ]:
H = '25,50,100,200,400,800,1800'
if os.path.exists('weights/lstm.weights.h5'):
    run(f'python src/common/evaluate.py {LSTM} --horizons {H}')
if os.path.exists('weights/tcn_checkpoint.pt'):
    run(f'python src/common/evaluate.py {TCN} --horizons {H}')

## 6. Per-file plots (actual vs predicted) + download

For each model, render one tall PNG per horizon (**100, 300, 600, 1800** steps) with
up to 50 test files stacked — actual ω vs predicted ω. The plots reuse the rollout
that already ran (no extra model calls), and are written to `outputs/`. The last cell
downloads every generated PNG to your machine.

In [ ]:
# Render the per-file plots (default horizons 100,300,600,1800). Override with
# --plot-horizons "100,300" if you want fewer / different ones.
if os.path.exists('weights/lstm.weights.h5'):
    run(f'python src/common/evaluate.py {LSTM} --plots')
if os.path.exists('weights/tcn_checkpoint.pt'):
    run(f'python src/common/evaluate.py {TCN} --plots')

# Download every generated PNG from Colab to your machine.
import glob
from google.colab import files

pngs = sorted(glob.glob('outputs/eval_plots_*.png'))
print(f'{len(pngs)} plot file(s):', *pngs, sep='\n  ')
for png in pngs:
    files.download(png)